# Voice Emotion Classification and Expression-Label Exploration

This academic notebook compares classical machine-learning models for vocal-emotion classification and a second **dataset-defined expression label**. The second task predicts the labels supplied with the recordings; it does not determine whether a speaker is truthful or what they genuinely feel.

The workflow splits original recordings before augmentation, keeps augmented versions grouped during cross-validation, and reserves untouched original recordings for final evaluation.

In [ ]:
%pip -q install librosa numpy pandas scikit-learn gradio joblib matplotlib

In [ ]:
from pathlib import Path
import math
import shutil
import zlib
import zipfile
from collections import Counter

import gradio as gr
import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


In [ ]:
# Google Colab expects the archive at /content/data.zip.
DATA_ZIP = Path("/content/data.zip")
EXTRACT_DIR = Path("/content/audio_dataset")
BASE_DIR = EXTRACT_DIR / "data"
SAMPLE_RATE = 22_050
TEST_SIZE = 0.20
RANDOM_STATE = 42
MODEL_DIR = Path("/content/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and prepare the dataset

The ZIP archive should contain `data/<emotion>_<expression-label>/*.wav`. Folder names provide two labels separated by the final underscore. The dataset is not included in this repository; use recordings you are allowed to process and share.

In [ ]:
def extract_dataset_safely(zip_path: Path, destination: Path) -> None:
    """Extract only WAV files under the expected data/ folder."""
    if not zip_path.exists():
        raise FileNotFoundError(
            f"Dataset archive not found: {zip_path}. Upload data.zip to the Colab session first."
        )

    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = Path(info.filename.replace("\\", "/")).parts
            if not parts or parts[0].lower() != "data":
                continue
            target = (destination.joinpath(*parts)).resolve()
            try:
                target.relative_to(root)
            except ValueError as exc:
                raise ValueError(f"Unsafe path in dataset archive: {info.filename}") from exc

            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            elif target.suffix.lower() == ".wav":
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open("wb") as output:
                    shutil.copyfileobj(source, output)

extract_dataset_safely(DATA_ZIP, EXTRACT_DIR)
if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Expected a data/ directory inside the archive. Looked for: {BASE_DIR}"
    )
print(f"Dataset extracted to: {BASE_DIR}")

In [ ]:
def collect_recordings(base_dir: Path) -> pd.DataFrame:
    rows = []
    for folder in sorted(base_dir.iterdir()):
        if not folder.is_dir():
            continue
        parts = folder.name.rsplit("_", maxsplit=1)
        if len(parts) != 2 or not all(parts):
            print(f"Skipping folder with an unexpected name: {folder.name}")
            continue

        emotion, expression_label = (part.strip().lower() for part in parts)
        for audio_path in sorted(folder.glob("*")):
            if audio_path.is_file() and audio_path.suffix.lower() == ".wav":
                rows.append({
                    "record_id": str(audio_path.resolve()),
                    "audio_path": str(audio_path),
                    "emotion": emotion,
                    "expression_label": expression_label,
                })

    records = pd.DataFrame(rows)
    if records.empty:
        raise ValueError(f"No WAV recordings found under {base_dir}")
    if records["record_id"].duplicated().any():
        raise ValueError("Duplicate recording paths were found.")
    return records

records = collect_recordings(BASE_DIR)
print(f"Original recordings: {len(records)}")
display(records.groupby(["emotion", "expression_label"]).size().rename("recordings").to_frame())

## 2. Split original recordings before augmentation

Each original WAV file belongs to only one split. If speaker IDs are available, use them as groups too so a speaker cannot appear in both training and test data.

In [ ]:
def split_original_recordings(records: pd.DataFrame, test_size: float = TEST_SIZE):
    joint_label = records["emotion"].astype(str) + "::" + records["expression_label"].astype(str)
    test_count = math.ceil(len(records) * test_size)
    train_count = len(records) - test_count
    joint_counts = joint_label.value_counts()

    if joint_counts.min() >= 2 and min(test_count, train_count) >= len(joint_counts):
        stratify = joint_label
        print("Using stratification by emotion/expression-label combination.")
    else:
        emotion_counts = records["emotion"].value_counts()
        if emotion_counts.min() >= 2 and min(test_count, train_count) >= len(emotion_counts):
            stratify = records["emotion"]
            print("Some label combinations are rare; stratifying by emotion only.")
        else:
            stratify = None
            print("Warning: not enough recordings for stratification; using a random split.")

    train_records, test_records = train_test_split(
        records, test_size=test_size, random_state=RANDOM_STATE, stratify=stratify
    )
    train_records = train_records.reset_index(drop=True)
    test_records = test_records.reset_index(drop=True)

    if not set(train_records["record_id"]).isdisjoint(test_records["record_id"]):
        raise RuntimeError("A source recording appears in both train and test splits.")
    for target in ("emotion", "expression_label"):
        missing = set(records[target]) - set(train_records[target])
        if missing:
            raise ValueError(f"Training split is missing {target} labels: {sorted(missing)}")
    return train_records, test_records

train_records, test_records = split_original_recordings(records)
print(f"Training source recordings: {len(train_records)}")
print(f"Held-out test source recordings: {len(test_records)}")

In [ ]:
FEATURE_NAMES = (
    [f"mfcc_{index:02d}" for index in range(1, 14)]
    + [f"chroma_{index:02d}" for index in range(1, 13)]
    + [f"spectral_contrast_{index:02d}" for index in range(1, 8)]
    + ["zero_crossing_rate"]
)

def extract_features_from_array(y: np.ndarray, sr: int) -> dict:
    y = np.asarray(y, dtype=np.float32)
    if y.size == 0:
        raise ValueError("Audio recording is empty.")
    if not np.isfinite(y).all():
        raise ValueError("Audio recording contains non-finite values.")
    if len(y) < 2048:
        y = librosa.util.fix_length(y, size=2048)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y=y)
    values = np.concatenate([
        np.mean(mfcc, axis=1),
        np.mean(chroma, axis=1),
        np.mean(contrast, axis=1),
        [float(np.mean(zcr))],
    ])
    return dict(zip(FEATURE_NAMES, values.astype(float)))

def load_audio(audio_path: str):
    return librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

In [ ]:
def augment_audio(y: np.ndarray, sr: int, record_id: str):
    """Create deterministic, modest variations of one training recording."""
    variants = [("original", y)]
    try:
        variants.append(("time_stretch", librosa.effects.time_stretch(y=y, rate=1.1)))
    except Exception as exc:
        print(f"Skipping time-stretch for one short/invalid clip: {exc}")
    try:
        variants.append(("pitch_shift", librosa.effects.pitch_shift(y=y, sr=sr, n_steps=2)))
    except Exception as exc:
        print(f"Skipping pitch-shift for one short/invalid clip: {exc}")

    stable_seed = (RANDOM_STATE + zlib.crc32(record_id.encode("utf-8"))) % (2**32)
    rng = np.random.default_rng(stable_seed)
    noise = rng.normal(0.0, 0.003, size=y.shape).astype(np.float32)
    variants.append(("noise", np.clip(y + noise, -1.0, 1.0)))
    return variants

def make_feature_frame(record_table: pd.DataFrame, augment: bool) -> pd.DataFrame:
    rows = []
    failed = []
    for item in record_table.itertuples(index=False):
        try:
            y, sr = load_audio(item.audio_path)
            variants = augment_audio(y, sr, item.record_id) if augment else [("original", y)]
            for variant_name, signal in variants:
                row = extract_features_from_array(signal, sr)
                row.update({
                    "record_id": item.record_id,
                    "variant": variant_name,
                    "emotion": item.emotion,
                    "expression_label": item.expression_label,
                })
                rows.append(row)
        except Exception as exc:
            failed.append((item.audio_path, str(exc)))

    if failed:
        print(f"Could not process {len(failed)} recording(s):")
        for audio_path, message in failed[:10]:
            print(f" - {audio_path}: {message}")
    frame = pd.DataFrame(rows)
    if frame.empty:
        raise ValueError("No usable audio features were extracted.")
    return frame

# Only training recordings are augmented. Test recordings remain original and untouched.
train_features = make_feature_frame(train_records, augment=True)
test_features = make_feature_frame(test_records, augment=False)
print(f"Training feature rows (including variants): {len(train_features)}")
print(f"Test feature rows (original recordings only): {len(test_features)}")

## 3. Select models with grouped cross-validation

All augmented versions of a source recording stay in the same fold. Model selection uses grouped cross-validated Macro-F1 on the training split. The held-out test split is reserved for final evaluation.

In [ ]:
def make_models():
    return {
        "Random Forest": lambda: RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ),
        "K-Nearest Neighbors": lambda: make_pipeline(
            StandardScaler(), KNeighborsClassifier(n_neighbors=3, weights="distance")
        ),
        "Logistic Regression": lambda: make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
        ),
    }

def compare_with_grouped_cv(frame: pd.DataFrame, target: str):
    X = frame[FEATURE_NAMES]
    y = frame[target].astype(str)
    groups = frame["record_id"]

    source_rows = frame[["record_id", target]].drop_duplicates()
    groups_per_class = source_rows.groupby(target)["record_id"].nunique()
    n_splits = min(5, int(groups_per_class.min()))
    if n_splits < 2:
        raise ValueError(
            f"Need at least two original recordings for every {target} class to run grouped CV. "
            "Collect more examples or merge/remove an unsupported label."
        )

    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    labels = sorted(y.unique())
    results = {}
    factories = make_models()
    for name, factory in factories.items():
        fold_scores = []
        for train_idx, validation_idx in cv.split(X, y, groups=groups):
            model = factory()
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            prediction = model.predict(X.iloc[validation_idx])
            fold_scores.append(f1_score(
                y.iloc[validation_idx], prediction, labels=labels, average="macro", zero_division=0
            ))
        results[name] = {"mean_macro_f1": float(np.mean(fold_scores)), "fold_scores": fold_scores}

    ranking = pd.DataFrame(
        {name: {"mean_macro_f1": values["mean_macro_f1"]} for name, values in results.items()}
    ).T.sort_values("mean_macro_f1", ascending=False)
    return ranking, factories


In [ ]:
model_bundles = {}
all_cv_results = {}

for target in ("emotion", "expression_label"):
    print(f"\n=== Task: {target} ===")
    ranking, factories = compare_with_grouped_cv(train_features, target)
    display(ranking)
    best_name = ranking.index[0]
    best_model = factories[best_name]()

    X_train = train_features[FEATURE_NAMES]
    y_train = train_features[target].astype(str)
    X_test = test_features[FEATURE_NAMES]
    y_test = test_features[target].astype(str)
    best_model.fit(X_train, y_train)
    prediction = best_model.predict(X_test)
    labels = sorted(set(y_train) | set(y_test))

    print(f"Selected using grouped CV: {best_name}")
    print(f"Held-out Macro-F1: {f1_score(y_test, prediction, labels=labels, average='macro', zero_division=0):.3f}")
    print(f"Held-out Weighted-F1: {f1_score(y_test, prediction, labels=labels, average='weighted', zero_division=0):.3f}")
    print(f"Held-out Balanced Accuracy: {balanced_accuracy_score(y_test, prediction):.3f}")
    print(classification_report(y_test, prediction, labels=labels, zero_division=0))

    display_labels = labels
    matrix = confusion_matrix(y_test, prediction, labels=display_labels)
    display = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=display_labels)
    fig, ax = plt.subplots(figsize=(7, 5))
    display.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
    ax.set_title(f"{target.replace('_', ' ').title()} - held-out test set")
    fig.tight_layout()
    plt.show()

    all_cv_results[target] = ranking
    model_bundles[target] = {
        "model": best_model,
        "model_name": best_name,
        "feature_names": FEATURE_NAMES,
        "sample_rate": SAMPLE_RATE,
        "classes": labels,
    }

### Interpreting the results

Macro-F1 gives each class equal weight, while weighted-F1 reflects class frequency. A small or imbalanced dataset can still produce unstable estimates. Report the number of original recordings and treat the held-out results as exploratory; if several recordings come from the same speaker, evaluate with speaker-level groups when those IDs are available.

In [ ]:
emotion_model_path = MODEL_DIR / "emotion_model.pkl"
expression_model_path = MODEL_DIR / "expression_label_model.pkl"
joblib.dump(model_bundles["emotion"], emotion_model_path)
joblib.dump(model_bundles["expression_label"], expression_model_path)
print(f"Saved: {emotion_model_path}")
print(f"Saved: {expression_model_path}")

## 4. Try a recording

The second output is the model's prediction of the dataset-provided expression label. It is not a lie detector and should not be interpreted as evidence about truthfulness or a person's internal emotional state.

In [ ]:
emotion_bundle = joblib.load(emotion_model_path)
expression_bundle = joblib.load(expression_model_path)

def extract_features_from_file(audio_path: str) -> pd.DataFrame:
    y, sr = load_audio(audio_path)
    values = extract_features_from_array(y, sr)
    return pd.DataFrame([values], columns=FEATURE_NAMES)

def predict_audio(audio_path):
    if audio_path is None:
        return "Record or upload audio first.", "No dataset-label prediction available."
    features = extract_features_from_file(audio_path)
    emotion = emotion_bundle["model"].predict(features)[0]
    expression_label = expression_bundle["model"].predict(features)[0]
    return str(emotion), str(expression_label)

interface = gr.Interface(
    fn=predict_audio,
    inputs=gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio recording"),
    outputs=[
        gr.Textbox(label="Predicted vocal-emotion label (experimental)"),
        gr.Textbox(label="Predicted dataset expression label (experimental)"),
    ],
    title="Voice Emotion Classification",
    description=(
        "Academic demonstration. Predictions reflect patterns in the supplied dataset; "
        "they do not reveal truthfulness or a speaker's internal state."
    ),
)
interface.launch()